In [11]:
from pathlib import Path
import sys, os

#RESUME_PATH = Path.home() / 'Desktop' /'Job Search'/'2026'/'axtria'/'Resume_Rajeev_Kulkarni_2026_03_11.pdf'
RESUME_PATH = Path.home() / "ml-explorations" / "job-search-copilot" / "data" / "resumes" / "resume.pdf"
REPO_ROOT = Path.home() / "ml-explorations" / "job-search-copilot"
SRC_PATH = REPO_ROOT / "src"
sys.path.insert(0, str(SRC_PATH))

DB_PATH = str(REPO_ROOT / "data" / "memory" / "copilot.db")
print(f"DB path: {DB_PATH}")
print(f"Exists: {Path(DB_PATH).exists()}")
print(f"Resume path: {RESUME_PATH}")
print(f"Exists: {RESUME_PATH.exists()}")
print(f"Readable: {os.access(RESUME_PATH, os.R_OK)}")

DB path: /Users/rajeevkulkarni/ml-explorations/job-search-copilot/data/memory/copilot.db
Exists: True
Resume path: /Users/rajeevkulkarni/ml-explorations/job-search-copilot/data/resumes/resume.pdf
Exists: True
Readable: True


In [2]:
import json, sqlite3

JSON_FIELDS = ["gap_analysis", "similar_companies", "similar_roles", 
               "live_openings", "company_competition", "company_alternatives"]

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()
cur.execute("SELECT * FROM jobs")
rows = cur.fetchall()

In [3]:
for row in rows:
    print(f"{row['company']} - {row['title']} | gap_analysis type: {type(row['gap_analysis']).__name__}")

SG Analytics - Associate Vice President Data Science  | gap_analysis type: str
Sigmoid - Director Data Science  | gap_analysis type: str
Axtria - Principal Decision Science | gap_analysis type: str
MathCo - Data Science Delivery Lead- Pharma | gap_analysis type: str
Intuituve.AI - Delivery Lead | gap_analysis type: str


In [4]:
analysis = json.loads(row["gap_analysis"])
print("Keys:", list(analysis.keys()))
print("Strengths count:", len(analysis.get("strengths", [])))
print("Gaps count:", len(analysis.get("gaps", [])))
print("First strength:", analysis.get("strengths", ["none"])[0][:100])

Keys: ['tailored_resume', 'strengths', 'gaps', 'similar_companies', 'similar_roles', 'live_openings_queries']
Strengths count: 6
Gaps count: 6
First strength: Deep healthcare domain expertise: 10+ years in payer/provider analytics with proven ability to work 


In [12]:
from pypdf import PdfReader
reader = PdfReader(RESUME_PATH)
resume_text = "\n".join(page.extract_text() or "" for page in reader.pages)
print(resume_text[:500])

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)


Details of Healthcare Projects (contd.) 
RAJEEV KULKARNI Principal Decision Scientist | Life Sciences & Pharma Analytics | Patient & RWD Analytics | Payer/Provider Insights | Agentic AI +91 7506115619 | kulkarni.rajeev@live.com | LinkedIn PROFILE SUMMARY Decision Science leader with 20+ years of experience, including 10+ years in healthcare analytics, 3 years dedicated to life sciences and pharma. Deep expertise in patient analytics (risk of readmission, mortality, etc.), oncology trials analyti


In [15]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.home() / "ml-explorations" / "job-search-copilot" / "src"))

from prompt1 import run_prompt1
from pypdf import PdfReader
import os, json, sqlite3

DB_PATH = str(Path.home() / "ml-explorations" / "job-search-copilot" / "data" / "memory" / "copilot.db")
RESUME_PATH = str(Path.home() / "ml-explorations" / "job-search-copilot" / "data" / "resumes" / "resume.pdf")
api_key = os.environ.get("ANTHROPIC_API_KEY")

print(f"resume exists: {Path(RESUME_PATH).exists()}")
print(f"api_key set: {bool(api_key)}")

resume exists: True
api_key set: True


In [17]:
from prompt1 import run_prompt1
from pypdf import PdfReader
import os, json, sqlite3
from pathlib import Path

RESUME_PATH = str(Path.home() / "path/to/your/Resume.pdf")  # update this
api_key = os.environ.get("ANTHROPIC_API_KEY")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()
cur.execute("SELECT * FROM jobs")
rows = cur.fetchall()
print(f"Total jobs to reanalyse: {len(rows)}")

for row in rows:
    jd_text = row["jd_text"] or ""
    if not jd_text:
        print(f"  SKIP {row['company']} — no JD text")
        continue
    try:
        print(f"  Analysing: {row['company']} — {row['title']}")
        analysis = run_prompt1(resume_text, jd_text, row["company"], api_key)
        cur.execute("""
            UPDATE jobs SET
                gap_analysis = ?,
                tailored_resume = ?,
                similar_companies = ?,
                similar_roles = ?,
                live_openings = ?
            WHERE job_id = ?
        """, (
            json.dumps(analysis),
            analysis.get("tailored_resume", ""),
            json.dumps(analysis.get("similar_companies", [])),
            json.dumps(analysis.get("similar_roles", [])),
            json.dumps(analysis.get("live_openings_queries", [])),
            row["job_id"]
        ))
        conn.commit()
        print(f"    Done — {len(analysis.get('strengths',[]))} strengths, {len(analysis.get('gaps',[]))} gaps")
    except Exception as e:
        print(f"    ERROR: {e}")

conn.close()
print("\nBatch reanalysis complete.")

Total jobs to reanalyse: 5
  Analysing: SG Analytics — Associate Vice President Data Science 
    Done — 6 strengths, 5 gaps
  Analysing: Sigmoid — Director Data Science 
    Done — 5 strengths, 4 gaps
  Analysing: Axtria — Principal Decision Science
    Done — 6 strengths, 3 gaps
  Analysing: MathCo — Data Science Delivery Lead- Pharma
    Done — 5 strengths, 3 gaps
  Analysing: Intuituve.AI — Delivery Lead
    Done — 6 strengths, 4 gaps

Batch reanalysis complete.
